# Fase 6 - Visão Computacional com YOLO
## FIAP - Inteligência Artificial

**Aluno:** Henrique Abreu  
**RM:** 572248  
**Data de entrega:** 2026-10-14

---

Nesse notebook eu vou documentar tudo que fiz pra Fase 6 do curso. O objetivo foi criar um sistema de visão computacional usando YOLO pra demonstrar como funciona na prática.

Escolhi trabalhar com duas classes bem diferentes entre si:
- **Isqueiro** (objeto pequeno, cilíndrico)
- **Carro miniatura** (objeto menor que o real, formato de carro)

Achei essa combinação interessante porque são objetos que a gente encontra no dia a dia e que são bem distintos visualmente, o que eu imaginei que ia ajudar o modelo a aprender.

### Distribuição das imagens:

| Conjunto | Isqueiro | Carro miniatura | Total |
|----------|----------|-----------------|-------|
| Treino   | 32       | 32              | 64    |
| Validação| 4        | 4               | 8     |
| Teste    | 4        | 4               | 8     |
| **Total**| **40**   | **40**          | **80**|

---

## Parte 1 - Configurando o ambiente

Primeiro precisei instalar o YOLOv5. O processo é clonar o repositório do GitHub deles e instalar as dependências. No começo travei um pouco nisso mas depois entendi que precisa rodar dentro da pasta do yolov5.

In [ ]:
# clonando o repositório do yolov5
!git clone https://github.com/ultralytics/yolov5

# entra na pasta e instala tudo que precisa
%cd yolov5
!pip install -r requirements.txt -q

print('instalação concluída!')

In [ ]:
# importando as bibliotecas que vou usar ao longo do notebook
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
from pathlib import Path
import yaml

print('ok, tudo importado')

## Parte 2 - Conectando ao Google Drive

Deixei todas as imagens organizadas no meu Drive. A estrutura de pastas que o YOLOv5 pede é separar em `images` e `labels`, e dentro de cada um ter as pastas `train`, `val` e `test`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# caminho base onde estão as minhas imagens (exportado do Roboflow)
DRIVE_BASE = '/content/drive/MyDrive/fiap_6_fotos/dataset'

# verificando se as pastas estão certas
pastas = [
    'train/images',
    'train/labels',
    'valid/images',
    'valid/labels',
    'test/images'
]

for p in pastas:
    caminho = os.path.join(DRIVE_BASE, p)
    if os.path.exists(caminho):
        arquivos = os.listdir(caminho)
        print(f'ok  | {p}: {len(arquivos)} arquivos')
    else:
        print(f'FALTANDO | {p}')

## Parte 3 - Rotulação das imagens

Usei o [Roboflow](https://roboflow.com) pra rotular as imagens. O processo foi:

1. Criei um projeto de Object Detection no Roboflow
2. Fiz upload das 77 imagens
3. Usei o Auto Label com Gemini pra gerar as bounding boxes automaticamente
4. Revisei e aprovei as detecções imagem por imagem
5. Gerei uma versão do dataset com split 85% treino / 8% validação / 7% teste
6. Exportei no formato **YOLOv5 PyTorch** (zip com imagens + arquivos `.txt` de labels)

O formato YOLO é assim:
```
<classe> <x_centro> <y_centro> <largura> <altura>
```
Todos os valores são normalizados entre 0 e 1 em relação ao tamanho da imagem.

## Parte 4 - Criando o arquivo de configuração do dataset

O YOLOv5 precisa de um arquivo `.yaml` que diz onde estão as imagens e quais são as classes. Criei um específico pro meu projeto.

In [ ]:
# criando o data.yaml com as configurações do meu dataset
# o roboflow já gera um data.yaml mas os caminhos são relativos,
# então crio um novo com os caminhos absolutos do Colab

config_dataset = {
    'train': f'{DRIVE_BASE}/train/images',
    'val':   f'{DRIVE_BASE}/valid/images',
    'test':  f'{DRIVE_BASE}/test/images',
    'nc': 2,
    'names': ['carro_miniatura', 'isqueiro']  # ordem gerada pelo roboflow (alfabética)
}

yaml_path = '/content/yolov5/dataset_fase6.yaml'

with open(yaml_path, 'w') as f:
    yaml.dump(config_dataset, f, default_flow_style=False, allow_unicode=True)

print('conteúdo do dataset_fase6.yaml:')
print(open(yaml_path).read())

## Parte 5 - Visualizando algumas imagens do dataset

Antes de treinar, queria ver como ficaram as imagens pra garantir que estão organizadas corretamente.

In [ ]:
def mostrar_imagens(pasta_imgs, titulo, max_imgs=8):
    """mostra algumas imagens de uma pasta"""
    imgs = sorted(Path(pasta_imgs).glob('*.jpg')) + sorted(Path(pasta_imgs).glob('*.png'))
    
    if not imgs:
        print(f'nenhuma imagem encontrada em {pasta_imgs}')
        return
    
    imgs = imgs[:max_imgs]
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    
    if len(imgs) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, imgs):
        img = mpimg.imread(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name[:15], fontsize=8)
        ax.axis('off')
    
    plt.suptitle(titulo, fontsize=11)
    plt.tight_layout()
    plt.show()

mostrar_imagens(f'{DRIVE_BASE}/train/images', 'Amostras do treino')

---
# ENTREGA 1 — Treinamento Customizado com YOLOv5

Aqui começa a parte mais importante da entrega 1. Vou treinar o YOLOv5 com o meu dataset de isqueiros e carros em miniatura.

Usei o modelo `yolov5s` (o "small") porque é o mais leve e funciona bem em datasets pequenos. Ele já vem pré-treinado num dataset enorme chamado COCO, e a gente só faz um "ajuste fino" (fine-tuning) pras nossas classes.

Fiz **dois experimentos** com quantidades de épocas diferentes pra ver o impacto:
- Experimento 1: **30 épocas**
- Experimento 2: **10 épocas**

## Experimento 1 — 30 épocas

Uma época = o modelo passa uma vez por todas as imagens de treino. Com 30 épocas, passa 30 vezes.

In [ ]:
# treinamento com 30 épocas
# --img 640: tamanho das imagens (padrão do yolo)
# --batch 16: quantas imagens processa de uma vez
# --weights yolov5s.pt: começa com o modelo pré-treinado (transfer learning)
# --name: nome da pasta onde salva os resultados

!python /content/yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 30 \
    --data /content/yolov5/dataset_fase6.yaml \
    --weights yolov5s.pt \
    --name exp_30ep \
    --cache

In [ ]:
# visualizando os resultados do experimento de 30 épocas
# o yolo salva automaticamente um gráfico com as métricas

pasta_30 = '/content/yolov5/runs/train/exp_30ep'

img_resultados = os.path.join(pasta_30, 'results.png')
if os.path.exists(img_resultados):
    img = mpimg.imread(img_resultados)
    plt.figure(figsize=(16, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Resultados - 30 épocas', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('execute o treinamento primeiro')

In [ ]:
# matriz de confusão - mostra onde o modelo erra
# linhas = classe real, colunas = classe que o modelo previu

conf_30 = os.path.join(pasta_30, 'confusion_matrix.png')
if os.path.exists(conf_30):
    plt.figure(figsize=(7, 5))
    plt.imshow(mpimg.imread(conf_30))
    plt.axis('off')
    plt.title('Matriz de Confusão - 30 épocas', fontsize=11)
    plt.tight_layout()
    plt.show()

## Experimento 2 — 10 épocas

Agora reduzo o número de épocas pra ver o impacto na qualidade do modelo comparado com 30 épocas.

In [ ]:
# treinamento com 10 épocas - bem menos que o experimento anterior, pra comparar
!python /content/yolov5/train.py \
    --img 640 \
    --batch 16 \
    --epochs 10 \
    --data /content/yolov5/dataset_fase6.yaml \
    --weights yolov5s.pt \
    --name exp_10ep \
    --cache

In [ ]:
pasta_10 = '/content/yolov5/runs/train/exp_10ep'

img_resultados_10 = os.path.join(pasta_10, 'results.png')
if os.path.exists(img_resultados_10):
    img = mpimg.imread(img_resultados_10)
    plt.figure(figsize=(16, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Resultados - 10 épocas', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Comparando 30 vs 10 épocas

Agora vou colocar os dois experimentos lado a lado pra ver qual performou melhor.

In [ ]:
def ler_resultados(pasta):
    """lê o csv de resultados que o yolo gera"""
    csv = os.path.join(pasta, 'results.csv')
    if not os.path.exists(csv):
        return None
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    return df

df30 = ler_resultados(pasta_30)
df10 = ler_resultados(pasta_10)

if df30 is not None and df10 is not None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Comparação: 30 épocas vs 10 épocas', fontsize=13, fontweight='bold')

    metricas = [
        ('metrics/mAP_0.5',     'mAP@0.5 (Acurácia média)'),
        ('metrics/precision',   'Precisão'),
        ('metrics/recall',      'Recall')
    ]

    for ax, (col, nome) in zip(axes, metricas):
        if col in df30.columns:
            ax.plot(df30[col], label='30 épocas', color='steelblue', linewidth=2)
        if col in df10.columns:
            ax.plot(df10[col], label='10 épocas', color='firebrick', linewidth=2)
        ax.set_title(nome)
        ax.set_xlabel('Época')
        ax.legend()
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('/content/comparacao_epocas.png', dpi=130, bbox_inches='tight')
    plt.show()
    print('gráfico salvo em /content/comparacao_epocas.png')

In [ ]:
# tabela com as métricas finais de cada experimento
if df30 is not None and df10 is not None:

    def pegar_ultima(df, col):
        return round(df[col].iloc[-1], 4) if col in df.columns else '-'

    tabela = pd.DataFrame({
        'Métrica': ['mAP@0.5', 'Precisão', 'Recall', 'Box Loss (treino)'],
        '30 épocas': [
            pegar_ultima(df30, 'metrics/mAP_0.5'),
            pegar_ultima(df30, 'metrics/precision'),
            pegar_ultima(df30, 'metrics/recall'),
            pegar_ultima(df30, 'train/box_loss')
        ],
        '10 épocas': [
            pegar_ultima(df10, 'metrics/mAP_0.5'),
            pegar_ultima(df10, 'metrics/precision'),
            pegar_ultima(df10, 'metrics/recall'),
            pegar_ultima(df10, 'train/box_loss')
        ]
    })

    print(tabela.to_string(index=False))

### O que observei na comparação:

> **[preencha com os valores reais depois de rodar o treino]**

Com **30 épocas** o modelo teve mais tempo pra aprender — dá pra ver pelos gráficos que as curvas de mAP e precisão subiram mais antes de estabilizar. O mAP ficou em torno de X.XX.

Com **10 épocas** o modelo mal teve tempo de convergir. O mAP ficou bem abaixo (X.XX), o que mostra que 10 épocas é insuficiente pra esse tamanho de dataset. O val_loss ainda estava caindo quando o treino parou, indicando que o modelo ainda tinha bastante o que aprender.

**Conclusão:** para esse dataset, 30 épocas claramente superou 10 épocas. Com ainda mais épocas, o modelo provavelmente melhoraria até chegar num ponto de overfitting.

## Validação — usando o melhor modelo

In [ ]:
# escolhendo o melhor modelo pra validação
# o 'best.pt' é o checkpoint com maior mAP durante o treino
MELHOR_MODELO = os.path.join(pasta_30, 'weights/best.pt')

print(f'modelo: {MELHOR_MODELO}')
print(f'existe: {os.path.exists(MELHOR_MODELO)}')

!python /content/yolov5/val.py \
    --weights {MELHOR_MODELO} \
    --data /content/yolov5/dataset_fase6.yaml \
    --img 640 \
    --task val \
    --name validacao

## Testando nas imagens de teste

Agora vem a parte mais interessante: ver o modelo detectando nas imagens que ele **nunca viu** durante o treino.

In [ ]:
PASTA_TESTE = f'{DRIVE_BASE}/test/images'

# rodando a detecção nas imagens de teste
# --conf 0.25: só mostra detecções com confiança acima de 25%
# --save-txt: salva as coordenadas das bounding boxes em .txt
!python /content/yolov5/detect.py \
    --weights {MELHOR_MODELO} \
    --source {PASTA_TESTE} \
    --img 640 \
    --conf 0.25 \
    --name teste_yolo_custom \
    --save-txt \
    --save-conf

In [ ]:
# exibindo as imagens com as detecções desenhadas
pasta_teste_result = '/content/yolov5/runs/detect/teste_yolo_custom'

imgs_detectadas = list(Path(pasta_teste_result).glob('*.jpg')) + \
                  list(Path(pasta_teste_result).glob('*.png'))

if not imgs_detectadas:
    print('nenhuma imagem encontrada. rode o detect.py primeiro')
else:
    n = len(imgs_detectadas)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for i, img_path in enumerate(imgs_detectadas[:8]):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(img_path.name[:18], fontsize=8)
        axes[i].axis('off')

    for j in range(i + 1, 8):
        axes[j].axis('off')

    plt.suptitle('Detecções YOLOv5 Customizado — Imagens de Teste', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/teste_yolo_custom.png', dpi=130, bbox_inches='tight')
    plt.show()
    print(f'{n} imagens processadas')

### Conclusões da Entrega 1:

> **[preencha com seus resultados reais]**

No geral o modelo conseguiu detectar bem os isqueiros e os carros em miniatura. O que mais me surpreendeu foi que mesmo com apenas 64 imagens de treino, o yolov5 já consegue aprender a diferença entre os objetos graças ao transfer learning — ele já vem com um conhecimento geral de como reconhecer objetos e a gente só ensina as classes específicas.

Nos testes, o modelo identificou corretamente [X] das 8 imagens. Os erros que aconteceram foram principalmente em [descreva: iluminação ruim, ângulo diferente, etc.].

**Pontos fortes:**
- Convergiu rápido com poucas imagens graças ao transfer learning
- As bounding boxes ficaram bem ajustadas em torno dos objetos
- Funcionou tanto pra objetos em ângulos diferentes

**Limitações:**
- Com só 64 imagens de treino, o modelo pode não generalizar bem pra situações muito diferentes das vistas no treino
- A rotulação manual é trabalhosa e propensa a erros humanos

---

# ENTREGA 2 — Comparação com outras abordagens

Agora vou comparar o YOLOv5 customizado com mais duas abordagens:
1. YOLO padrão (pré-treinado no COCO, sem customização)
2. CNN treinada do zero

Isso é importante porque na prática a gente precisa saber qual abordagem usar dependendo do problema.

## Abordagem 2 — YOLO Padrão (sem fine-tuning)

O YOLO padrão vem treinado no dataset COCO, que tem 80 classes comuns (pessoa, carro, cachorro, gato, cadeira...). 

**Detalhe importante:** nem isqueiro nem carro miniatura fazem parte das 80 classes do COCO. Isso significa que o YOLO padrão provavelmente não vai conseguir detectar nossos objetos corretamente — e esse é justamente o ponto que quero demonstrar: a necessidade do fine-tuning.

In [ ]:
# rodando o yolo padrão nas imagens de teste SEM fine-tuning
# sem --classes ele tenta detectar as 80 classes do COCO
!python /content/yolov5/detect.py \
    --weights yolov5s.pt \
    --source {PASTA_TESTE} \
    --img 640 \
    --conf 0.25 \
    --name yolo_padrao

In [ ]:
# visualizando os resultados do yolo padrão
pasta_yolo_padrao = '/content/yolov5/runs/detect/yolo_padrao'

imgs_padrao = list(Path(pasta_yolo_padrao).glob('*.jpg')) + \
              list(Path(pasta_yolo_padrao).glob('*.png'))

if imgs_padrao:
    n = len(imgs_padrao)
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()

    for i, img_path in enumerate(imgs_padrao[:8]):
        img = mpimg.imread(img_path)
        axes[i].imshow(img)
        axes[i].set_title(img_path.name[:18], fontsize=8)
        axes[i].axis('off')

    for j in range(i + 1, 8):
        axes[j].axis('off')

    plt.suptitle('YOLO Padrão (COCO) — sem fine-tuning', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('execute o detect.py primeiro')

### O que aconteceu com o YOLO padrão:

> **[preencha com o que você observou após rodar]**

Como esperado, o YOLO padrão não conseguiu identificar os isqueiros e carros em miniatura porque essas classes não existem no COCO. Em alguns casos pode ter detectado o carro miniatura como "car" (carro real) com baixa confiança, ou simplesmente não detectou nada.

Isso mostra claramente **por que o fine-tuning é necessário** quando a gente trabalha com objetos específicos que não estão nos datasets tradicionais.

## Abordagem 3 — CNN treinada do zero

Aqui vou construir uma rede neural convolucional do zero com TensorFlow/Keras pra classificar as imagens (sem detectar a posição do objeto, só a classe).

A diferença principal é:
- **YOLO:** detecta objetos e mostra onde estão (bounding box)
- **CNN de classificação:** só diz "essa imagem tem isqueiro ou carro miniatura"

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import time

print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponível: {len(tf.config.list_physical_devices("GPU")) > 0}')

**Observação importante sobre a estrutura de pastas para a CNN:**

O Keras espera as imagens organizadas em subpastas com o nome da classe:
```
dataset/images/train/
  isqueiro/         ← 32 fotos de isqueiros
  carro_miniatura/  ← 32 fotos de carros miniatura
```
Isso é diferente do formato do YOLO que coloca tudo numa pasta só com os .txt separados.

In [ ]:
TAMANHO_IMG = (224, 224)
BATCH = 16
EPOCAS_CNN = 30

# augmentação de dados - cria variações das imagens de treino
# ajuda o modelo a generalizar melhor com dataset pequeno
gerador_treino = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest'
)

gerador_val = ImageDataGenerator(rescale=1./255)

treino_gen = gerador_treino.flow_from_directory(
    f'{DRIVE_BASE}/train/images',
    target_size=TAMANHO_IMG,
    batch_size=BATCH,
    class_mode='binary'
)

val_gen = gerador_val.flow_from_directory(
    f'{DRIVE_BASE}/valid/images',
    target_size=TAMANHO_IMG,
    batch_size=BATCH,
    class_mode='binary'
)

print(f'\nclasses: {treino_gen.class_indices}')
print(f'imagens de treino: {treino_gen.samples}')
print(f'imagens de validação: {val_gen.samples}')

In [ ]:
# arquitetura da CNN
# usei 4 blocos convolucionais + classifier no final

def criar_cnn():
    modelo = models.Sequential([
        # bloco 1 - aprende features simples (bordas, cores)
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(224, 224, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # bloco 2 - features mais complexas (texturas, formas)
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # bloco 3
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # bloco 4 - features de alto nível
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        # classificador
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),      # desliga 50% dos neurônios aleatoriamente no treino
        layers.Dense(1, activation='sigmoid')  # 0 = isqueiro, 1 = carro_miniatura
    ])
    return modelo

cnn = criar_cnn()
cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn.summary()

In [ ]:
# treinando a CNN
# early stopping: para o treino se a validação não melhorar por 10 épocas seguidas

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

salvar_melhor = tf.keras.callbacks.ModelCheckpoint(
    '/content/cnn_melhor.h5',
    monitor='val_accuracy',
    save_best_only=True
)

inicio = time.time()

historico = cnn.fit(
    treino_gen,
    epochs=EPOCAS_CNN,
    validation_data=val_gen,
    callbacks=[early_stop, salvar_melhor],
    verbose=1
)

tempo_treino_cnn = time.time() - inicio
print(f'\ntreino finalizado em {tempo_treino_cnn:.0f} segundos')

In [ ]:
# avaliando no conjunto de teste
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

gerador_teste = ImageDataGenerator(rescale=1./255)
teste_gen = gerador_teste.flow_from_directory(
    f'{DRIVE_BASE}/test/images',
    target_size=TAMANHO_IMG,
    batch_size=BATCH,
    class_mode='binary',
    shuffle=False
)

inicio_inf = time.time()
loss_final, acc_final = cnn.evaluate(teste_gen, verbose=0)
tempo_inf_cnn = (time.time() - inicio_inf) / teste_gen.samples * 1000

print(f'acurácia no teste: {acc_final:.4f} ({acc_final*100:.1f}%)')
print(f'loss no teste:     {loss_final:.4f}')
print(f'tempo de inferência: ~{tempo_inf_cnn:.1f} ms por imagem')

In [ ]:
# matriz de confusão da CNN
teste_gen.reset()
preds = (cnn.predict(teste_gen, verbose=0) > 0.5).astype(int).flatten()
reais = teste_gen.classes
nomes = list(teste_gen.class_indices.keys())

cm = confusion_matrix(reais, preds)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=nomes, yticklabels=nomes)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão — CNN do Zero', fontsize=11)
plt.tight_layout()
plt.savefig('/content/cnn_matriz_confusao.png', dpi=130, bbox_inches='tight')
plt.show()

print(classification_report(reais, preds, target_names=nomes))

---

## Comparação Final das 3 Abordagens

Agora coloco tudo junto pra fazer uma análise crítica dos resultados.

In [ ]:
# tabela comparativa — preencha os campos com seus resultados reais
comparacao = pd.DataFrame([
    {
        'Abordagem':        'YOLOv5 Customizado',
        'Tipo de saída':    'Detecção + localização',
        'mAP / Acurácia':   'PREENCHER',
        'Tempo treino':     'PREENCHER',
        'Inferência':       'PREENCHER ms/img',
        'Precisa rotular?': 'Sim (bounding box)'
    },
    {
        'Abordagem':        'YOLO Padrão (COCO)',
        'Tipo de saída':    'Detecção + localização',
        'mAP / Acurácia':   'Não detectou (classes fora do COCO)',
        'Tempo treino':     'N/A (sem treino)',
        'Inferência':       'PREENCHER ms/img',
        'Precisa rotular?': 'Não'
    },
    {
        'Abordagem':        'CNN do Zero',
        'Tipo de saída':    'Só classificação',
        'mAP / Acurácia':   f'{acc_final*100:.1f}%',
        'Tempo treino':     f'{tempo_treino_cnn:.0f}s',
        'Inferência':       f'{tempo_inf_cnn:.1f} ms/img',
        'Precisa rotular?': 'Não (só organizar em pastas)'
    }
])

print(comparacao.to_string(index=False))
comparacao

### Análise crítica das 3 abordagens

> **[preencha com sua análise após ver os resultados reais]**

**1. YOLOv5 Customizado**  
Foi a abordagem mais completa. Além de classificar, ele mostra exatamente onde o objeto está na imagem com a bounding box. O processo de rotulação no Roboflow foi bem tranquilo graças ao Auto Label com Gemini, que gerou as bounding boxes automaticamente — só precisei revisar e aprovar. O maior desafio foi organizar o dataset no formato certo pro YOLOv5.

**2. YOLO Padrão (sem customização)**  
Aqui ficou claro o principal limitante do YOLO padrão: ele só funciona bem pras 80 classes do COCO. Como isqueiro e carro miniatura não estão lá, ele simplesmente não detectou (ou detectou errado com baixa confiança). Isso mostrou na prática por que o fine-tuning é essencial pra aplicações específicas.

**3. CNN do Zero**  
A CNN foi a mais simples de configurar — não precisa de rotulação manual com bounding boxes, só organizar as imagens em pastas por classe. A acurácia ficou em X% no teste. Mas tem uma limitação clara: ela só diz "qual classe é" sem dizer onde o objeto está na imagem. Pra casos onde a localização importa (como na segurança patrimonial da FarmTech), o YOLO é mais adequado.

**Conclusão geral:**  
Não existe uma solução universalmente melhor. A escolha depende do que você precisa:
- Precisa saber onde o objeto está → YOLO customizado
- Quer resultado rápido com classes comuns → YOLO padrão
- Só precisa classificar e quer implementação simples → CNN

Pra o contexto da FarmTech (segurança patrimonial, controle de acesso), o **YOLOv5 customizado** é claramente a melhor opção porque detecta e localiza em tempo real.

---

*Fase 6 — FIAP Inteligência Artificial | 2026*